In [1]:
!pip install nuscenes-devkit
!pip install pyquaternion
!pip install torch torchvision
!apt-get install -y unzip

'apt-get' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
!pip install "numpy>=1.22.0,<2.0.0"

In [3]:

# INSTALL DEPENDENCIES


!pip install nuscenes-devkit pyquaternion
!apt-get install -y unzip

print("Dependencies installed.")

Dependencies installed.


'apt-get' is not recognized as an internal or external command,
operable program or batch file.


In [4]:

# DOWNLOAD FULL NUSCENES MINI DATA

import os

 

# Download metadata
!wget -nc https://www.nuscenes.org/data/v1.0-mini.tgz
!tar -xzf v1.0-mini.tgz

# Download samples (images + lidar)
!wget -nc https://www.nuscenes.org/data/v1.0-mini-samples.tgz
!tar -xzf v1.0-mini-samples.tgz

# Download sweeps (required for lidar)
!wget -nc https://www.nuscenes.org/data/v1.0-mini-sweeps.tgz
!tar -xzf v1.0-mini-sweeps.tgz

print("NuScenes mini fully downloaded.")


'wget' is not recognized as an internal or external command,
operable program or batch file.


tar: Error opening archive: Failed to open 'v1.0-mini.tgz'
'wget' is not recognized as an internal or external command,
operable program or batch file.
tar: Error opening archive: Failed to open 'v1.0-mini-samples.tgz'
'wget' is not recognized as an internal or external command,
operable program or batch file.


NuScenes mini fully downloaded.


tar: Error opening archive: Failed to open 'v1.0-mini-sweeps.tgz'


In [ ]:
"""
OmniVec2  –  Stage 1 Masked Pretraining (RGB only, NuScenes)
=============================================================
Paper : OmniVec2 – A Novel Multimodal Multitask Network
Focus : Unimodal self-supervised masked pretraining of the
        feature-transformation network  f(·)  on NuScenes RGB images.

Architecture (per paper §3.1 & §3.2):
    Tokenizer  →  Encoder  f(·)  →  [discard after training] Decoder
    • Tokenizer  : Conv2d patch embedding + sinusoidal positional encoding
    • Encoder f(·): BERT-style transformer + 3 FC layers with ReLU
    • Decoder     : lightweight transformer  + linear head  →  pixel patches
    • Target      : raw normalised pixel patches (per-patch zero-mean, unit-var)
    • Loss        : MSE (ℓ₂) on **masked** patches only
"""

# ──────────────────────── 0. Imports ────────────────────────
import os
import math
import random
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

from nuscenes.nuscenes import NuScenes

# ──────────────────────── 1. Config ─────────────────────────
# --- data ---
NUSCENES_DATAROOT = "/content"          # Colab default
NUSCENES_VERSION  = "v1.0-mini"         # change to "v1.0-trainval" for full
CAMERAS = [
    "CAM_FRONT", "CAM_FRONT_LEFT", "CAM_FRONT_RIGHT",
    "CAM_BACK",  "CAM_BACK_LEFT",  "CAM_BACK_RIGHT",
]

# --- image / patch ---
IMG_SIZE   = 224          # standard ViT resolution
PATCH_SIZE = 16           # → 14×14 = 196 patches
NUM_PATCHES = (IMG_SIZE // PATCH_SIZE) ** 2   # 196

# --- model ---
EMBED_DIM  = 256          # encoder hidden dim
NUM_HEADS  = 8            # must divide EMBED_DIM
ENC_LAYERS = 6            # transformer blocks in encoder
DEC_EMBED_DIM = 128       # lighter decoder
DEC_HEADS  = 4
DEC_LAYERS = 4
FC_HIDDEN  = 128          # intermediate FC dim after encoder (paper §3.1)

# --- masking & training ---
MASK_RATIO = 0.75         # user's choice
EPOCHS     = 100
LR         = 1e-4
BATCH_SIZE = 64
NUM_WORKERS = 2
SEED       = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ──────────────────────── 2. Helpers ────────────────────────
def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


def patchify(imgs: torch.Tensor) -> torch.Tensor:
    """
    imgs : (B, 3, H, W)  →  patches : (B, N, patch_dim)
    where patch_dim = 3 * PATCH_SIZE ** 2
    """
    p = PATCH_SIZE
    B, C, H, W = imgs.shape
    assert H == W == IMG_SIZE
    h = w = H // p
    x = imgs.reshape(B, C, h, p, w, p)
    x = x.permute(0, 2, 4, 1, 3, 5)          # (B, h, w, C, p, p)
    x = x.reshape(B, h * w, C * p * p)       # (B, N, patch_dim)
    return x


def unpatchify(patches: torch.Tensor) -> torch.Tensor:
    """
    patches : (B, N, patch_dim)  →  imgs : (B, 3, H, W)
    """
    p = PATCH_SIZE
    h = w = IMG_SIZE // p
    B, N, D = patches.shape
    C = 3
    x = patches.reshape(B, h, w, C, p, p)
    x = x.permute(0, 3, 1, 4, 2, 5)          # (B, C, h, p, w, p)
    x = x.reshape(B, C, h * p, w * p)
    return x


def normalize_patches(patches: torch.Tensor):
    """
    Per-patch zero-mean, unit-variance normalization (paper §3.2.2).
    Returns normalised patches, mean, var  (all same shape).
    """
    mean  = patches.mean(dim=-1, keepdim=True)
    var   = patches.var(dim=-1, keepdim=True)
    patches_norm = (patches - mean) / (var + 1e-6).sqrt()
    return patches_norm, mean, var


# ──────────────────────── 3. Dataset ────────────────────────
class NuScenesRGBDataset(Dataset):
    """
    Each item = one camera image from one sample.
    Scene-based train / val split to avoid data leakage.
    """

    def __init__(self, nusc: NuScenes, scene_tokens: set):
        super().__init__()
        self.nusc = nusc
        self.items = []  # list of (sample_token, camera_name)

        for scene in nusc.scene:
            if scene["token"] not in scene_tokens:
                continue
            sample_token = scene["first_sample_token"]
            while sample_token:
                sample = nusc.get("sample", sample_token)
                for cam in CAMERAS:
                    self.items.append((sample["data"][cam], cam))
                sample_token = sample["next"] if sample["next"] != "" else None

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        cam_data_token, cam_name = self.items[idx]
        cam_data = self.nusc.get("sample_data", cam_data_token)
        img_path = os.path.join(NUSCENES_DATAROOT, cam_data["filename"])

        img = Image.open(img_path).convert("RGB")
        img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)

        # to tensor, scale to [0, 1]
        img_np = np.array(img, dtype=np.float32) / 255.0
        img_tensor = torch.from_numpy(img_np).permute(2, 0, 1)  # (3, H, W)
        return img_tensor


def build_dataloaders(nusc: NuScenes, split_ratio: float = 0.8):
    """
    Scene-based split → DataLoaders.
    """
    all_scenes = nusc.scene.copy()
    random.shuffle(all_scenes)
    n_train = int(len(all_scenes) * split_ratio)

    train_tokens = {s["token"] for s in all_scenes[:n_train]}
    val_tokens   = {s["token"] for s in all_scenes[n_train:]}

    train_ds = NuScenesRGBDataset(nusc, train_tokens)
    val_ds   = NuScenesRGBDataset(nusc, val_tokens)

    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=NUM_WORKERS,
                          pin_memory=True, drop_last=True)
    val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS,
                          pin_memory=True)

    print(f"Train scenes: {len(train_tokens)}  |  Val scenes: {len(val_tokens)}")
    print(f"Train images: {len(train_ds)}      |  Val images: {len(val_ds)}")
    return train_dl, val_dl


# ──────────────────────── 4. Positional Encoding ────────────
def get_2d_sincos_pos_embed(embed_dim: int, grid_size: int) -> torch.Tensor:
    """
    2-D sine-cosine positional embeddings  →  (1, grid_size**2, embed_dim).
    Standard approach from MAE / ViT.
    """
    grid_h = torch.arange(grid_size, dtype=torch.float32)
    grid_w = torch.arange(grid_size, dtype=torch.float32)
    grid   = torch.stack(torch.meshgrid(grid_h, grid_w, indexing="ij"), dim=-1)  # (G, G, 2)
    grid   = grid.reshape(-1, 2)  # (N, 2)

    half_dim = embed_dim // 2
    omega = 1.0 / (10000 ** (torch.arange(0, half_dim, 2, dtype=torch.float32) / half_dim))

    out_h = grid[:, 0:1] * omega.unsqueeze(0)  # (N, half_dim//2)
    out_w = grid[:, 1:2] * omega.unsqueeze(0)

    pe = torch.cat([torch.sin(out_h), torch.cos(out_h),
                    torch.sin(out_w), torch.cos(out_w)], dim=-1)  # (N, 2*half_dim)
    if pe.shape[-1] < embed_dim:
        pe = F.pad(pe, (0, embed_dim - pe.shape[-1]))
    pe = pe[:, :embed_dim]
    return pe.unsqueeze(0)   # (1, N, D)


# ──────────────────────── 5. Network Components ─────────────

# ---- 5a  Patch Embedding (Tokenizer) -----------------------
class PatchEmbedding(nn.Module):
    """
    Paper §3.1 — modality-specific tokenizer.
    Conv2d projects each patch to EMBED_DIM; add positional encoding.
    """
    def __init__(self):
        super().__init__()
        self.proj = nn.Conv2d(3, EMBED_DIM,
                              kernel_size=PATCH_SIZE, stride=PATCH_SIZE)
        grid_size = IMG_SIZE // PATCH_SIZE
        self.pos_embed = nn.Parameter(
            get_2d_sincos_pos_embed(EMBED_DIM, grid_size),
            requires_grad=False,                       # fixed sin-cos
        )

    def forward(self, imgs: torch.Tensor) -> torch.Tensor:
        """imgs : (B, 3, H, W)  →  tokens: (B, N, EMBED_DIM)"""
        x = self.proj(imgs)                             # (B, D, h, w)
        x = x.flatten(2).transpose(1, 2)                # (B, N, D)
        x = x + self.pos_embed                          # add pos
        return x


# ---- 5b  Encoder  f(·) -------------------------------------
class Encoder(nn.Module):
    """
    Paper §3.1 — Feature transformation network f(·):
        BERT-style transformer  →  3 FC layers with ReLU (dim reduction).
    """
    def __init__(self):
        super().__init__()
        # Transformer layers
        enc_layer = nn.TransformerEncoderLayer(
            d_model=EMBED_DIM,
            nhead=NUM_HEADS,
            dim_feedforward=EMBED_DIM * 4,
            activation="gelu",          # paper: "GeLU activation prior to the MLP layer"
            batch_first=True,
            norm_first=True,            # pre-norm (modern default)
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=ENC_LAYERS)
        self.norm = nn.LayerNorm(EMBED_DIM)

        # 3 FC layers with ReLU — paper §3.1:
        # "three fully connected layers with ReLU activation …
        #  to reduce the dimensions of the features"
        self.fc = nn.Sequential(
            nn.Linear(EMBED_DIM, FC_HIDDEN),
            nn.ReLU(),
            nn.Linear(FC_HIDDEN, FC_HIDDEN),
            nn.ReLU(),
            nn.Linear(FC_HIDDEN, FC_HIDDEN),
            nn.ReLU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x : (B, N_vis, EMBED_DIM)  →  (B, N_vis, FC_HIDDEN)"""
        x = self.transformer(x)
        x = self.norm(x)
        x = self.fc(x)
        return x


# ---- 5c  Decoder -------------------------------------------
class Decoder(nn.Module):
    """
    Lightweight decoder for masked pretraining (discarded after Stage 1).
    Predicts **raw pixel patches** (per-patch normalised).
    """
    def __init__(self):
        super().__init__()
        self.patch_dim = 3 * PATCH_SIZE ** 2   # output dim per patch

        # Learnable mask token
        self.mask_token = nn.Parameter(torch.zeros(1, 1, DEC_EMBED_DIM))
        nn.init.normal_(self.mask_token, std=0.02)

        # Positional embedding for full sequence
        grid_size = IMG_SIZE // PATCH_SIZE
        self.dec_pos_embed = nn.Parameter(
            get_2d_sincos_pos_embed(DEC_EMBED_DIM, grid_size),
            requires_grad=False,
        )

        # Project encoder output to decoder dim
        self.enc_to_dec = nn.Linear(FC_HIDDEN, DEC_EMBED_DIM)

        # Transformer layers
        dec_layer = nn.TransformerEncoderLayer(
            d_model=DEC_EMBED_DIM,
            nhead=DEC_HEADS,
            dim_feedforward=DEC_EMBED_DIM * 4,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(dec_layer, num_layers=DEC_LAYERS)
        self.norm = nn.LayerNorm(DEC_EMBED_DIM)

        # Final projection → pixel patches
        self.head = nn.Linear(DEC_EMBED_DIM, self.patch_dim)

    def forward(self, encoded: torch.Tensor,
                ids_restore: torch.Tensor) -> torch.Tensor:
        """
        encoded     : (B, N_vis, FC_HIDDEN)
        ids_restore : (B, N)                   (from random_mask)
        returns     : (B, N, patch_dim)         predicted pixel patches
        """
        # project to decoder dim
        x = self.enc_to_dec(encoded)                            # (B, N_vis, D_dec)

        B, N_vis, D = x.shape
        N = ids_restore.shape[1]                                # total patches

        # append mask tokens for masked positions
        mask_tokens = self.mask_token.expand(B, N - N_vis, -1)  # (B, N_mask, D_dec)
        x = torch.cat([x, mask_tokens], dim=1)                  # (B, N, D_dec)

        # un-shuffle to original patch order
        x = torch.gather(x, 1, ids_restore.unsqueeze(-1).expand(-1, -1, D))

        # add positional embeddings
        x = x + self.dec_pos_embed

        # transformer + norm
        x = self.transformer(x)
        x = self.norm(x)

        # predict pixel patches
        x = self.head(x)                                         # (B, N, patch_dim)
        return x


# ──────────────────────── 6. Random Masking ──────────────────
def random_mask(tokens: torch.Tensor, mask_ratio: float = MASK_RATIO):
    """
    Randomly mask a fraction of tokens (same as MAE).
    Returns:
        x_visible   (B, N_vis, D)
        ids_restore (B, N)       – indices to unshuffle
        ids_keep    (B, N_vis)   – indices of kept patches
        mask        (B, N)       – binary mask (1=masked, 0=visible)
    """
    B, N, D = tokens.shape
    len_keep = int(N * (1 - mask_ratio))

    noise       = torch.rand(B, N, device=tokens.device)
    ids_shuffle = torch.argsort(noise, dim=1)
    ids_restore = torch.argsort(ids_shuffle, dim=1)

    ids_keep = ids_shuffle[:, :len_keep]
    x_visible = torch.gather(tokens, 1,
                             ids_keep.unsqueeze(-1).expand(-1, -1, D))

    # binary mask: 1 = masked, 0 = visible
    mask = torch.ones(B, N, device=tokens.device)
    mask.scatter_(1, ids_keep, 0.0)

    return x_visible, ids_restore, ids_keep, mask


# ──────────────────────── 7. Stage 1 Model ──────────────────
class OmniVec2Stage1(nn.Module):
    """
    Full Stage-1 masked autoencoder for images.
    After pretraining, discard self.decoder and keep
    self.patch_embed + self.encoder  as the pretrained f(·).
    """
    def __init__(self):
        super().__init__()
        self.patch_embed = PatchEmbedding()
        self.encoder     = Encoder()
        self.decoder     = Decoder()

    def forward(self, imgs: torch.Tensor):
        """
        imgs   : (B, 3, H, W)
        returns: pred_patches (B, N, patch_dim),
                 target_patches (B, N, patch_dim)   — normalised,
                 mask (B, N)                         — 1 = masked
        """
        # --- tokenize ---
        tokens = self.patch_embed(imgs)                  # (B, N, D)

        # --- mask ---
        x_vis, ids_restore, ids_keep, mask = random_mask(tokens, MASK_RATIO)

        # --- encode (only visible tokens) ---
        latent = self.encoder(x_vis)                     # (B, N_vis, FC_HIDDEN)

        # --- decode (reconstruct all patches) ---
        pred_patches = self.decoder(latent, ids_restore) # (B, N, patch_dim)

        # --- reconstruction target: raw patches, per-patch normalised ---
        with torch.no_grad():
            raw_patches = patchify(imgs)                 # (B, N, patch_dim)
            target_patches, _, _ = normalize_patches(raw_patches)

        return pred_patches, target_patches, mask

    def save_encoder(self, path: str):
        """Save only the pretrained encoder f(·) = patch_embed + encoder."""
        state = {
            "patch_embed": self.patch_embed.state_dict(),
            "encoder":     self.encoder.state_dict(),
        }
        torch.save(state, path)
        print(f"Encoder saved → {path}")


# ──────────────────────── 8. Loss ────────────────────────────
def masked_mse_loss(pred: torch.Tensor,
                    target: torch.Tensor,
                    mask: torch.Tensor) -> torch.Tensor:
    """
    Paper §3.2.2: ℓ₂ loss on masked patches only.
    pred, target : (B, N, patch_dim)
    mask         : (B, N)   1 = masked (compute loss), 0 = visible
    """
    loss_per_patch = ((pred - target) ** 2).mean(dim=-1)    # (B, N)
    loss = (loss_per_patch * mask).sum() / mask.sum()
    return loss


# ──────────────────────── 9. Metrics (optional) ──────────────
@torch.no_grad()
def reconstruction_accuracy(pred: torch.Tensor,
                            target: torch.Tensor,
                            mask: torch.Tensor) -> float:
    """
    Relative reconstruction accuracy on *masked* patches.
    acc = 1 − ‖pred − target‖ / (‖target‖ + eps)
    """
    eps = 1e-8
    diff_norm   = torch.norm(pred - target, dim=-1)         # (B, N)
    target_norm = torch.norm(target, dim=-1) + eps
    acc_per_patch = 1.0 - diff_norm / target_norm            # (B, N)
    # average only over masked patches
    acc = (acc_per_patch * mask).sum() / mask.sum()
    return acc.item()


# ──────────────────────── 10. Training ───────────────────────
def train_one_epoch(model, dataloader, optimizer):
    model.train()
    total_loss, total_acc, n = 0.0, 0.0, 0

    for imgs in dataloader:
        imgs = imgs.to(DEVICE)
        pred, target, mask = model(imgs)

        loss = masked_mse_loss(pred, target, mask)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        total_acc  += reconstruction_accuracy(pred, target, mask) * imgs.size(0)
        n += imgs.size(0)

    return total_loss / n, total_acc / n


@torch.no_grad()
def validate(model, dataloader):
    model.eval()
    total_loss, total_acc, n = 0.0, 0.0, 0

    for imgs in dataloader:
        imgs = imgs.to(DEVICE)
        pred, target, mask = model(imgs)

        loss = masked_mse_loss(pred, target, mask)

        total_loss += loss.item() * imgs.size(0)
        total_acc  += reconstruction_accuracy(pred, target, mask) * imgs.size(0)
        n += imgs.size(0)

    return total_loss / n, total_acc / n


def train(model, train_dl, val_dl, epochs=EPOCHS, lr=LR):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr,
                                  weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-6
    )

    history = {"train_loss": [], "val_loss": [],
               "train_acc":  [], "val_acc":  []}

    print("\n========== STAGE 1 — MASKED PRETRAINING ==========\n")
    for epoch in range(1, epochs + 1):
        t_loss, t_acc = train_one_epoch(model, train_dl, optimizer)
        v_loss, v_acc = validate(model, val_dl)
        scheduler.step()

        history["train_loss"].append(t_loss)
        history["val_loss"].append(v_loss)
        history["train_acc"].append(t_acc)
        history["val_acc"].append(v_acc)

        print(f"Epoch {epoch:3d}/{epochs} | "
              f"Train Loss {t_loss:.4f} | "
              f"Val Loss {v_loss:.4f} | "
              f"LR {scheduler.get_last_lr()[0]:.2e}")

    print("\n✓ Training complete.\n")
    return history


# ──────────────────────── 11. Visualisation ──────────────────
def plot_curves(history: dict):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(history["train_loss"], label="Train", marker="o", markersize=4)
    ax1.plot(history["val_loss"],   label="Val",   marker="o", markersize=4)
    ax1.set_title("Stage 1 — Loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("MSE Loss")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(history["train_acc"], label="Train", marker="o", markersize=4)
    ax2.plot(history["val_acc"],   label="Val",   marker="o", markersize=4)
    ax2.set_title("Stage 1 — Reconstruction Accuracy")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


@torch.no_grad()
def visualise_reconstruction(model, dataloader, n_samples: int = 4):
    """
    Show original  |  masked  |  reconstruction  for a few images.
    """
    model.eval()
    imgs = next(iter(dataloader))[:n_samples].to(DEVICE)
    pred, target, mask = model(imgs)

    # un-normalise predicted patches (approximate – uses global stats)
    raw_patches = patchify(imgs)
    mean  = raw_patches.mean(dim=-1, keepdim=True)
    var   = raw_patches.var(dim=-1, keepdim=True)
    pred_pixels = pred * (var + 1e-6).sqrt() + mean

    # build reconstruction: visible patches from original, masked from prediction
    recon_patches = raw_patches.clone()
    mask_expanded = mask.unsqueeze(-1).expand_as(recon_patches).bool()
    recon_patches[mask_expanded] = pred_pixels[mask_expanded]

    recon_imgs  = unpatchify(recon_patches)

    # build masked image (grey out masked patches)
    masked_patches = raw_patches.clone()
    masked_patches[mask_expanded] = 0.5  # grey
    masked_imgs = unpatchify(masked_patches)

    fig, axes = plt.subplots(n_samples, 3, figsize=(12, 4 * n_samples))
    titles = ["Original", "Masked", "Reconstruction"]

    for i in range(n_samples):
        for j, img_t in enumerate([imgs[i], masked_imgs[i], recon_imgs[i]]):
            img_np = img_t.cpu().clamp(0, 1).permute(1, 2, 0).numpy()
            ax = axes[i, j] if n_samples > 1 else axes[j]
            ax.imshow(img_np)
            ax.set_title(titles[j])
            ax.axis("off")

    plt.tight_layout()
    plt.show()


# ──────────────────────── 12. Main ───────────────────────────
def main():
    print(f"Device : {DEVICE}")
    print(f"Config : IMG_SIZE={IMG_SIZE}, PATCH_SIZE={PATCH_SIZE}, "
          f"EMBED_DIM={EMBED_DIM}, MASK_RATIO={MASK_RATIO}")
    print(f"         ENC_LAYERS={ENC_LAYERS}, DEC_LAYERS={DEC_LAYERS}")
    print()

    # ---- load NuScenes ----
    nusc = NuScenes(version=NUSCENES_VERSION,
                    dataroot=NUSCENES_DATAROOT, verbose=True)

    # ---- dataloaders (scene-based split) ----
    train_dl, val_dl = build_dataloaders(nusc)

    # ---- model ----
    model = OmniVec2Stage1().to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"Model parameters : {n_params:.2f} M\n")

    # ---- train ----
    history = train(model, train_dl, val_dl)

    # ---- plots ----
    plot_curves(history)
    visualise_reconstruction(model, val_dl)

    # ---- save encoder ----
    model.save_encoder("omnivec2_stage1_encoder.pth")

    return model, history


if __name__ == "__main__":
    model, history = main()


Device : cuda
Config : IMG_SIZE=224, PATCH_SIZE=16, EMBED_DIM=256, MASK_RATIO=0.75
         ENC_LAYERS=6, DEC_LAYERS=4

Loading NuScenes tables for version v1.0-mini...
23 category,
8 attribute,
4 visibility,
911 instance,
12 sensor,
120 calibrated_sensor,
31206 ego_pose,
8 log,
10 scene,
404 sample,
31206 sample_data,
18538 sample_annotation,
4 map,
Done loading in 0.417 seconds.
Reverse indexing ...
Done reverse indexing in 0.1 seconds.
Train scenes: 8  |  Val scenes: 2
Train images: 1950      |  Val images: 474


/tmp/ipython-input-3669578750.py:251: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(enc_layer, num_layers=ENC_LAYERS)
/tmp/ipython-input-3669578750.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(dec_layer, num_layers=DEC_LAYERS)


Model parameters : 5.99 M


========== STAGE 1 — MASKED PRETRAINING ==========

Epoch   1/100 | Train Loss 1.1550 | Val Loss 1.0544 | LR 1.00e-04
Epoch   2/100 | Train Loss 1.0283 | Val Loss 1.0035 | LR 9.99e-05
Epoch   3/100 | Train Loss 1.0010 | Val Loss 0.9943 | LR 9.98e-05
Epoch   4/100 | Train Loss 0.9950 | Val Loss 0.9933 | LR 9.96e-05
Epoch   5/100 | Train Loss 0.9921 | Val Loss 0.9954 | LR 9.94e-05
Epoch   6/100 | Train Loss 0.9889 | Val Loss 1.0002 | LR 9.91e-05
Epoch   7/100 | Train Loss 0.9845 | Val Loss 1.0016 | LR 9.88e-05
Epoch   8/100 | Train Loss 0.9790 | Val Loss 1.0023 | LR 9.84e-05
Epoch   9/100 | Train Loss 0.9725 | Val Loss 1.0124 | LR 9.80e-05
Epoch  10/100 | Train Loss 0.9661 | Val Loss 1.0041 | LR 9.76e-05
Epoch  11/100 | Train Loss 0.9616 | Val Loss 1.0065 | LR 9.71e-05
Epoch  12/100 | Train Loss 0.9580 | Val Loss 0.9951 | LR 9.65e-05
Epoch  13/100 | Train Loss 0.9560 | Val Loss 0.9948 | LR 9.59e-05
Epoch  14/100 | Train Loss 0.9533 | Val Loss 0.9907 | LR 9.53e